In [35]:
import numpy as np
import cv2

from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
painting = "/content/painting.jpg"
painting2 = "/content/painting2.jpg"
painting3 = "/content/painting3.png"
mask1 = "/content/mask1.jpg"
mask2 = "/content/mask2.jpg"


In [38]:
def preprocess_image(path):

    img = Image.open(path).convert("RGB")

    img = img.resize((128,128))

    img = np.array(img)

    img = img / 255.0   # normalize

    img = np.expand_dims(img, axis=0)

    return img

In [39]:
cnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(128,128,3)),

    tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu')  # feature vector
])

In [40]:
cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [41]:
dummy = np.zeros((1,128,128,3))
cnn.predict(dummy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step


array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
      dtype=float32)

In [42]:
def import_features(image_path):

    img = preprocess_image(image_path)

    features = cnn.predict(img)

    return features

In [43]:
painting_features_cnn = import_features(painting)
mask1_features_cnn = import_features(mask1)
mask2_features_cnn = import_features(mask2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


In [44]:
from sklearn.metrics.pairwise import cosine_similarity

painting_feat = import_features(painting)
painting2_feat = import_features(painting2)
painting3_feat = import_features(painting3)
mask1_feat = import_features(mask1)
mask2_feat = import_features(mask2)

sim1 = cosine_similarity(painting_feat, mask1_feat)
sim2 = cosine_similarity(painting_feat, mask2_feat)
sim3 = cosine_similarity(painting2_feat, mask1_feat)
sim4 = cosine_similarity(painting2_feat, mask2_feat)
sim5 = cosine_similarity(painting3_feat, mask1_feat)
sim6 = cosine_similarity(painting3_feat, mask2_feat)

print("Painting vs Mask1:", sim1[0][0])
print("Painting vs Mask2:", sim2[0][0])
print("Painting2 vs Mask1:", sim3[0][0])
print("Painting2 vs Mask2:", sim4[0][0])
print("Painting3 vs Mask1:", sim5[0][0])
print("Painting3 vs Mask2:", sim6[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Painting vs Mask1: 0.9196199
Painting vs Mask2: 0.9229524
Painting2 vs Mask1: 0.9294738
Painting2 vs Mask2: 0.9328095
Painting3 vs Mask1: 0.87158644
Painting3 vs Mask2: 0.8844563


In [45]:
model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

In [46]:
def extract_features(img_path):

    # Load image
    img = image.load_img(img_path, target_size=(224,224))

    # Convert to array
    img_array = image.img_to_array(img)

    # Expand dimensions
    img_array = np.expand_dims(img_array, axis=0)

    # Preprocess for ResNet
    img_array = preprocess_input(img_array)

    # Extract features
    features = model.predict(img_array)

    return features


In [47]:
painting_features = extract_features(painting)
painting2_features = extract_features(painting2)
painting3_features = extract_features(painting3)
mask1_features = extract_features(mask1)
mask2_features = extract_features(mask2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step


In [48]:
similarity_mask1 = cosine_similarity(painting_features, mask1_features)
similarity_mask2 = cosine_similarity(painting_features, mask2_features)

similarity_mask3 = cosine_similarity(painting2_features, mask1_features)
similarity_mask4 = cosine_similarity(painting2_features, mask2_features)

similarity_mask5 = cosine_similarity(painting3_features, mask1_features)
similarity_mask6 = cosine_similarity(painting3_features, mask2_features)

In [49]:
# Print results
print("Similarity between Painting1 and Mask1:", similarity_mask1[0][0])
print("Similarity between Painting1 and Mask2:", similarity_mask2[0][0])
# Print results
print("Similarity between Painting2 and Mask1:", similarity_mask3[0][0])
print("Similarity between Painting2 and Mask2:", similarity_mask4[0][0])
# Print results
print("Similarity between Painting3 and Mask1:", similarity_mask5[0][0])
print("Similarity between Painting3 and Mask2:", similarity_mask6[0][0])

Similarity between Painting1 and Mask1: 0.23160046
Similarity between Painting1 and Mask2: 0.27229327
Similarity between Painting2 and Mask1: 0.54537284
Similarity between Painting2 and Mask2: 0.6076473
Similarity between Painting3 and Mask1: 0.39417583
Similarity between Painting3 and Mask2: 0.46698272


In [50]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [51]:
import torch


# Function to extract embedding
def get_embedding(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = model.vision_model(**inputs)

    # Extract pooled embedding
    embedding = outputs.pooler_output

    return embedding.cpu().numpy()


In [52]:
# Get embeddings
painting_emb = get_embedding(painting)
painting2_emb = get_embedding(painting2)
painting3_emb = get_embedding(painting3)
mask1_emb = get_embedding(mask1)
mask2_emb = get_embedding(mask2)

In [53]:
# Compute similarity
sim1 = cosine_similarity(painting_emb, mask1_emb)
sim2 = cosine_similarity(painting_emb, mask2_emb)
sim3 = cosine_similarity(painting2_emb, mask1_emb)
sim4 = cosine_similarity(painting2_emb, mask2_emb)
sim5 = cosine_similarity(painting3_emb, mask1_emb)
sim6 = cosine_similarity(painting3_emb, mask2_emb)


In [54]:
print("Similarity (Painting1 vs Mask1):", sim1[0][0])
print("Similarity (Painting1 vs Mask2):", sim2[0][0])
print("Similarity (Painting2 vs Mask1):", sim3[0][0])
print("Similarity (Painting2 vs Mask2):", sim4[0][0])
print("Similarity (Painting3 vs Mask1):", sim5[0][0])
print("Similarity (Painting3 vs Mask2):", sim6[0][0])

Similarity (Painting1 vs Mask1): 0.46147
Similarity (Painting1 vs Mask2): 0.44774178
Similarity (Painting2 vs Mask1): 0.43513623
Similarity (Painting2 vs Mask2): 0.43276688
Similarity (Painting3 vs Mask1): 0.5038273
Similarity (Painting3 vs Mask2): 0.538597
